In [19]:
import pandas as pd
import numpy as np
import os
import sys
import json
import re
import time
import datetime
import random
import string
import requests
import matplotlib.pyplot as plt
from helpers import *

In [20]:
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
visits = pd.read_csv('./data/visits.csv')
display(visits.head())
#print(visits.info())
#pmh = pd.read_csv('./data/pmh.csv')
#display(pmh.head())
#labs = pd.read_csv('./data/labs.csv')
#display(labs.head())
#meds = pd.read_csv('./data/meds.csv')
#display(meds.head())

,MRN,CSN,Visit_no,Visits,Age,Gender,Race,Ethnicity,Means_of_arrival,Triage_Temp,Triage_HR,Triage_RR,Triage_SpO2,Triage_SBP,Triage_DBP,Triage_acuity,CC,ED_dispo,Hours_to_next_visit,Dispo_class_next_visit,ED_LOS,Hosp_LOS,DC_dispo,Payor_class,Admit_service,Dx_ICD9,Dx_ICD10,Dx_name,Arrival_time,Roomed_time,Dispo_time,Admit_time,Departure_time
0,98880961,98874959,1,1,82,F,White,Non-Hispanic/Non-Latino,EMS,36.2,81.0,18.0,96.0,122.0,86.0,3-Urgent,FALL,Inpatient,NaN,NaN,6.13,1.0,Skilled Nursing Facility,Medicare,Medicine,873.20,S01.21XA,"Laceration of nose, initial encounter",2283-03-02T07:36:59Z,2283-03-02T07:36:59Z,2283-03-02T12:29:59Z,2283-03-02T12:29:59Z,2283-03-02T13:44:59Z
1,99121566,99354408,1,2,59,F,Other,Non-Hispanic/Non-Latino,EMS,36.8,98.0,14.0,98.0,103.0,66.0,2-Emergent,"ALTERED MENTAL STATUS,SHORTNESS OF BREATH",Inpatient,4841.0,ICU,9.53,7.0,Skilled Nursing Facility,NaN,Emergency Medicine,780.97,R41.82,"Altered mental status, unspecified altered men...",2239-08-22T01:13:51Z,2239-08-22T01:16:51Z,2239-08-22T07:20:51Z,2239-08-22T07:20:51Z,2239-08-22T10:45:51Z
2,99121566,99121037,2,2,63,F,Other,Non-Hispanic/Non-Latino,EMS,37.2,96.0,18.0,98.0,91.0,61.0,3-Urgent,"ABNORMAL LAB,GENERAL WEAKNESS",ICU,NaN,NaN,10.52,16.0,Skilled Nursing Facility,NaN,Critical Care,"038.9, 785.52, 995.92","A41.9, R65.21",Septic shock (CMS-HCC),2240-03-11T03:06:51Z,2240-03-11T03:43:51Z,2240-03-11T08:31:51Z,2240-03-11T08:31:51Z,2240-03-11T13:37:51Z
3,99608739,99327434,1,2,90,F,White,Non-Hispanic/Non-Latino,Self,NaN,69.0,16.0,98.0,107.0,42.0,3-Urgent,POSSIBLE DVT,Discharge,6704.0,Discharge,4.38,NaN,Home/Work (includes foster care),Medicare,NaN,729.81,M79.89,Leg swelling,2296-08-31T16:43:05Z,2296-08-31T17:56:05Z,2296-08-31T20:56:05Z,NaN,2296-08-31T21:06:05Z
4,99608739,99502864,2,2,89,F,White,Non-Hispanic/Non-Latino,Self,36.4,65.0,17.0,98.0,103.0,57.0,3-Urgent,URINARY RETENTION,Discharge,NaN,Discharge,4.45,NaN,Home/Work (includes foster care),Medicare,NaN,788.20,R33.9,Urinary retention,2297-06-07T05:08:05Z,2297-06-07T05:59:05Z,2297-06-07T08:55:05Z,NaN,2297-06-07T09:35:05Z


In [21]:
'''group1_vars = ['subject_id', 'hadm_id', 'stay_id',
               'anchor_age', 'age', 'gender', 'race', 'ethnicity', 'insurance',
               'intime', 'outtime', 'anchor_year', 'in_year', 'edregtime', 'edouttime', 'admittime', 'dischtime', 'intime_icu', 'outtime_icu', 'dod', 'deathtime', 
               'ed_los', 'ed_los_hours', 'next_ed_visit_time', 'next_ed_visit_time_diff', 'next_ed_visit_time_diff_days',
               'time_to_icu_transfer_hours', 'time_death_discharge', 'time_death_ed_depart', 'time_death_ed_arrival',
               'triage_temperature', 'triage_heartrate', 'triage_resprate', 'triage_o2sat', 'triage_sbp', 'triage_dbp', 'triage_pain', 'triage_acuity',
               'triage_gcs', 'triage_sofa', 'triage_MAP', 
               'ed_temperature_last', 'ed_heartrate_last', 'ed_resprate_last', 'ed_o2sat_last', 'ed_sbp_last', 'ed_dbp_last', 'ed_pain_last', 'ed_acuity_last',
               'ed_gcs_last',
               'lab']'''
group1_vars = ['subject_id', 'stay_id',
               'age', 'gender', 'race', 'ethnicity',
               'intime', 'outtime', 'in_year', 'edregtime', 'edouttime', 'admittime', 'dischtime', 'intime_icu', 'outtime_icu',
               'ed_los', 'ed_los_hours', 'next_ed_visit_time', 'next_ed_visit_time_diff', 'next_ed_visit_time_diff_days',
               'time_to_icu_transfer_hours',
               'triage_temperature', 'triage_heartrate', 'triage_resprate', 'triage_o2sat', 'triage_sbp', 'triage_dbp', 'triage_acuity',
               'triage_MAP']
group1_df = pd.DataFrame(columns=group1_vars)
display(group1_df)

,subject_id,stay_id,age,gender,race,ethnicity,intime,outtime,in_year,edregtime,edouttime,admittime,dischtime,intime_icu,outtime_icu,ed_los,ed_los_hours,next_ed_visit_time,next_ed_visit_time_diff,next_ed_visit_time_diff_days,time_to_icu_transfer_hours,triage_temperature,triage_heartrate,triage_resprate,triage_o2sat,triage_sbp,triage_dbp,triage_acuity,triage_MAP


### 1. ID

In [22]:
group1_df['subject_id'] = visits['MRN']
#group1_df['hadm_id'] = visits['Admit_time'].notna()
group1_df['stay_id'] = visits['CSN']
display(group1_df.head())

,subject_id,stay_id,age,gender,race,ethnicity,intime,outtime,in_year,edregtime,edouttime,admittime,dischtime,intime_icu,outtime_icu,ed_los,ed_los_hours,next_ed_visit_time,next_ed_visit_time_diff,next_ed_visit_time_diff_days,time_to_icu_transfer_hours,triage_temperature,triage_heartrate,triage_resprate,triage_o2sat,triage_sbp,triage_dbp,triage_acuity,triage_MAP
0,98880961,98874959,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,99121566,99354408,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,99121566,99121037,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,99608739,99327434,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,99608739,99502864,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### 2. Demo

In [23]:
#group1_df['anchor_age'] = visits['Age']
group1_df['age'] = visits['Age']
group1_df['gender'] = visits['Gender']
group1_df['race'] = visits['Race']
group1_df['ethnicity'] = visits['Ethnicity']
#group1_df['insurance'] = 'Unavailable'

display(group1_df.head())

,subject_id,stay_id,age,gender,race,ethnicity,intime,outtime,in_year,edregtime,edouttime,admittime,dischtime,intime_icu,outtime_icu,ed_los,ed_los_hours,next_ed_visit_time,next_ed_visit_time_diff,next_ed_visit_time_diff_days,time_to_icu_transfer_hours,triage_temperature,triage_heartrate,triage_resprate,triage_o2sat,triage_sbp,triage_dbp,triage_acuity,triage_MAP
0,98880961,98874959,82,F,White,Non-Hispanic/Non-Latino,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,99121566,99354408,59,F,Other,Non-Hispanic/Non-Latino,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,99121566,99121037,63,F,Other,Non-Hispanic/Non-Latino,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,99608739,99327434,90,F,White,Non-Hispanic/Non-Latino,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,99608739,99502864,89,F,White,Non-Hispanic/Non-Latino,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### 3. Time

In [24]:
def adjust_time_by_subtracting_year(time_series):
    year = pd.to_numeric(time_series.str[:4], errors='coerce')
    month_day_time = time_series.str[4:]
    new_year = year - 200
    new_time_str = new_year.astype(str).str.replace('.0', '', regex=False) + month_day_time
    new_time_str[time_series.isna()] = np.nan 
    return pd.to_datetime(new_time_str).dt.tz_localize(None)

In [25]:
group1_df['intime'] = adjust_time_by_subtracting_year(visits['Arrival_time'])
group1_df['outtime'] = adjust_time_by_subtracting_year(visits['Departure_time'])

#group1_df['anchor_year'] = visits['Arrival_time'].str[:4].astype(int)
group1_df['in_year'] = visits['Arrival_time'].str[:4].astype(int)

group1_df['edregtime'] = adjust_time_by_subtracting_year(visits['Roomed_time'])
group1_df['edouttime'] = adjust_time_by_subtracting_year(visits['Dispo_time'])

group1_df['admittime'] = adjust_time_by_subtracting_year(visits[visits['ED_dispo'] == 'Inpatient']['Admit_time'])
group1_df['intime_icu'] = adjust_time_by_subtracting_year(visits[visits['ED_dispo'] == 'ICU']['Admit_time'])

hosp_los_timedelta = pd.to_timedelta(visits['Hosp_LOS'], unit='D')
group1_df['dischtime'] = group1_df['admittime'] + hosp_los_timedelta
group1_df['outtime_icu'] = group1_df['intime_icu'] + hosp_los_timedelta

#group1_df['dod'] = pd.NaT
#group1_df['deathtime'] = pd.NaT

In [26]:
group1_df['ed_los'] = round(visits['ED_LOS'] / 24, 1)
group1_df['ed_los_hours'] = visits['ED_LOS']

visit_interval = pd.to_timedelta(visits['Hours_to_next_visit'], unit='h')
group1_df['next_ed_visit_time'] = group1_df['intime'] + visit_interval
group1_df['next_ed_visit_time_diff'] = visits['Hours_to_next_visit']
group1_df['next_ed_visit_time_diff_days'] = round(visits['Hours_to_next_visit'] / 24, 1)

time_diff_to_icu = group1_df['intime_icu'] - group1_df['intime']
group1_df['time_to_icu_transfer_hours'] = round(time_diff_to_icu.dt.total_seconds() / 3600, 2)

#group1_df['time_death_discharge'] = group1_df['deathtime'] - group1_df['dischtime']
#group1_df['time_death_ed_depart'] = group1_df['deathtime'] - group1_df['outtime']
#group1_df['time_death_ed_arrival'] = group1_df['deathtime'] - group1_df['intime']

display(group1_df)

,subject_id,stay_id,age,gender,race,ethnicity,intime,outtime,in_year,edregtime,edouttime,admittime,dischtime,intime_icu,outtime_icu,ed_los,ed_los_hours,next_ed_visit_time,next_ed_visit_time_diff,next_ed_visit_time_diff_days,time_to_icu_transfer_hours,triage_temperature,triage_heartrate,triage_resprate,triage_o2sat,triage_sbp,triage_dbp,triage_acuity,triage_MAP
0,98880961,98874959,82,F,White,Non-Hispanic/Non-Latino,2083-03-02 07:36:59,2083-03-02 13:44:59,2283,2083-03-02 07:36:59,2083-03-02 12:29:59,2083-03-02 12:29:59,2083-03-03 12:29:59,NaT,NaT,0.3,6.13,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,99121566,99354408,59,F,Other,Non-Hispanic/Non-Latino,2039-08-22 01:13:51,2039-08-22 10:45:51,2239,2039-08-22 01:16:51,2039-08-22 07:20:51,2039-08-22 07:20:51,2039-08-29 07:20:51,NaT,NaT,0.4,9.53,2040-03-10 18:13:51,4841.0,201.7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,99121566,99121037,63,F,Other,Non-Hispanic/Non-Latino,2040-03-11 03:06:51,2040-03-11 13:37:51,2240,2040-03-11 03:43:51,2040-03-11 08:31:51,NaT,NaT,2040-03-11 08:31:51,2040-03-27 08:31:51,0.4,10.52,NaT,NaN,NaN,5.42,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,99608739,99327434,90,F,White,Non-Hispanic/Non-Latino,2096-08-31 16:43:05,2096-08-31 21:06:05,2296,2096-08-31 17:56:05,2096-08-31 20:56:05,NaT,NaT,NaT,NaT,0.2,4.38,2097-06-07 00:43:05,6704.0,279.3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,99608739,99502864,89,F,White,Non-Hispanic/Non-Latino,2097-06-07 05:08:05,2097-06-07 09:35:05,2297,2097-06-07 05:59:05,2097-06-07 08:55:05,NaT,NaT,NaT,NaT,0.2,4.45,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
118380,99648524,99789968,41,F,Other,Non-Hispanic/Non-Latino,2022-09-23 02:03:34,2022-09-23 06:38:34,2222,2022-09-23 02:58:34,2022-09-23 05:46:34,NaT,NaT,NaT,NaT,0.2,4.58,2022-09-25 21:03:34,67.0,2.8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
118381,99312340,99391376,37,M,Black or African American,Non-Hispanic/Non-Latino,2045-12-31 02:58:40,2045-12-31 06:30:40,2245,2045-12-31 03:52:40,2045-12-31 05:10:40,2045-12-31 05:10:40,2046-01-01 05:10:40,NaT,NaT,0.1,3.53,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
118382,99828357,99836129,22,M,Other,Hispanic/Latino,2034-09-07 02:27:43,2034-09-07 06:22:43,2234,2034-09-07 04:27:43,2034-09-07 05:04:43,NaT,NaT,NaT,NaT,0.2,3.92,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
118383,98964925,98979879,24,M,Other,Hispanic/Latino,2088-11-06 10:18:50,2088-11-06 14:19:50,2288,2088-11-06 11:22:50,2088-11-06 13:11:50,NaT,NaT,NaT,NaT,0.2,4.02,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### 4. Vital

In [27]:
group1_df['triage_temperature'] = visits['Triage_Temp']
group1_df['triage_heartrate'] = visits['Triage_HR']
group1_df['triage_resprate'] = visits['Triage_RR']
group1_df['triage_o2sat'] = visits['Triage_SpO2']
group1_df['triage_sbp'] = visits['Triage_SBP']
group1_df['triage_dbp'] = visits['Triage_DBP']
#group1_df['triage_pain'] = 0
group1_df['triage_acuity'] = visits['Triage_acuity'].str[0].replace('n', np.nan).astype('Int64')
#group1_df['triage_gcs'] = 0
#group1_df['triage_sofa'] = 0
group1_df['triage_MAP'] = round((group1_df['triage_sbp'] + 2 * group1_df['triage_dbp']) / 3, 1)

#group1_df['ed_temperature_last'] = visits['Triage_Temp']
#group1_df['ed_heartrate_last'] = visits['Triage_HR']
#group1_df['ed_resprate_last'] = visits['Triage_RR']
#group1_df['ed_o2sat_last'] = visits['Triage_SpO2']
#group1_df['ed_sbp_last'] = visits['Triage_SBP']
#group1_df['ed_dbp_last'] = visits['Triage_DBP']
#group1_df['ed_pain_last'] = 0
#group1_df['ed_acuity_last'] = visits['Triage_acuity'].str[0].replace('n', np.nan).fillna(0).astype('Int64')
#group1_df['ed_gcs_last'] = 0
display(group1_df)

,subject_id,stay_id,age,gender,race,ethnicity,intime,outtime,in_year,edregtime,edouttime,admittime,dischtime,intime_icu,outtime_icu,ed_los,ed_los_hours,next_ed_visit_time,next_ed_visit_time_diff,next_ed_visit_time_diff_days,time_to_icu_transfer_hours,triage_temperature,triage_heartrate,triage_resprate,triage_o2sat,triage_sbp,triage_dbp,triage_acuity,triage_MAP
0,98880961,98874959,82,F,White,Non-Hispanic/Non-Latino,2083-03-02 07:36:59,2083-03-02 13:44:59,2283,2083-03-02 07:36:59,2083-03-02 12:29:59,2083-03-02 12:29:59,2083-03-03 12:29:59,NaT,NaT,0.3,6.13,NaT,NaN,NaN,NaN,36.2,81.0,18.0,96.0,122.0,86.0,3,98.0
1,99121566,99354408,59,F,Other,Non-Hispanic/Non-Latino,2039-08-22 01:13:51,2039-08-22 10:45:51,2239,2039-08-22 01:16:51,2039-08-22 07:20:51,2039-08-22 07:20:51,2039-08-29 07:20:51,NaT,NaT,0.4,9.53,2040-03-10 18:13:51,4841.0,201.7,NaN,36.8,98.0,14.0,98.0,103.0,66.0,2,78.3
2,99121566,99121037,63,F,Other,Non-Hispanic/Non-Latino,2040-03-11 03:06:51,2040-03-11 13:37:51,2240,2040-03-11 03:43:51,2040-03-11 08:31:51,NaT,NaT,2040-03-11 08:31:51,2040-03-27 08:31:51,0.4,10.52,NaT,NaN,NaN,5.42,37.2,96.0,18.0,98.0,91.0,61.0,3,71.0
3,99608739,99327434,90,F,White,Non-Hispanic/Non-Latino,2096-08-31 16:43:05,2096-08-31 21:06:05,2296,2096-08-31 17:56:05,2096-08-31 20:56:05,NaT,NaT,NaT,NaT,0.2,4.38,2097-06-07 00:43:05,6704.0,279.3,NaN,NaN,69.0,16.0,98.0,107.0,42.0,3,63.7
4,99608739,99502864,89,F,White,Non-Hispanic/Non-Latino,2097-06-07 05:08:05,2097-06-07 09:35:05,2297,2097-06-07 05:59:05,2097-06-07 08:55:05,NaT,NaT,NaT,NaT,0.2,4.45,NaT,NaN,NaN,NaN,36.4,65.0,17.0,98.0,103.0,57.0,3,72.3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
118380,99648524,99789968,41,F,Other,Non-Hispanic/Non-Latino,2022-09-23 02:03:34,2022-09-23 06:38:34,2222,2022-09-23 02:58:34,2022-09-23 05:46:34,NaT,NaT,NaT,NaT,0.2,4.58,2022-09-25 21:03:34,67.0,2.8,NaN,36.8,86.0,18.0,100.0,134.0,76.0,2,95.3
118381,99312340,99391376,37,M,Black or African American,Non-Hispanic/Non-Latino,2045-12-31 02:58:40,2045-12-31 06:30:40,2245,2045-12-31 03:52:40,2045-12-31 05:10:40,2045-12-31 05:10:40,2046-01-01 05:10:40,NaT,NaT,0.1,3.53,NaT,NaN,NaN,NaN,36.9,89.0,17.0,96.0,145.0,81.0,2,102.3
118382,99828357,99836129,22,M,Other,Hispanic/Latino,2034-09-07 02:27:43,2034-09-07 06:22:43,2234,2034-09-07 04:27:43,2034-09-07 05:04:43,NaT,NaT,NaT,NaT,0.2,3.92,NaT,NaN,NaN,NaN,36.8,85.0,14.0,99.0,144.0,97.0,2,112.7
118383,98964925,98979879,24,M,Other,Hispanic/Latino,2088-11-06 10:18:50,2088-11-06 14:19:50,2288,2088-11-06 11:22:50,2088-11-06 13:11:50,NaT,NaT,NaT,NaT,0.2,4.02,NaT,NaN,NaN,NaN,NaN,75.0,16.0,100.0,122.0,78.0,4,92.7


In [28]:
'''triage_cols = [col for col in group1_df.columns if col.startswith('triage_')]
triage_cols_missing_ratio = group1_df[triage_cols].isna().mean() * 100
print(f"Missing value percentage for triage_ columns: {triage_cols_missing_ratio}")
for col in triage_cols:
    median_val = group1_df[col].median()
    group1_df[col] = group1_df[col].fillna(median_val)'''

'triage_cols = [col for col in group1_df.columns if col.startswith(\'triage_\')]\ntriage_cols_missing_ratio = group1_df[triage_cols].isna().mean() * 100\nprint(f"Missing value percentage for triage_ columns: {triage_cols_missing_ratio}")\nfor col in triage_cols:\n    median_val = group1_df[col].median()\n    group1_df[col] = group1_df[col].fillna(median_val)'

In [31]:
display(group1_df)
group1_df.to_csv('./results/group1_master_dataset.csv', index=False)

,subject_id,stay_id,age,gender,race,ethnicity,intime,outtime,in_year,edregtime,edouttime,admittime,dischtime,intime_icu,outtime_icu,ed_los,ed_los_hours,next_ed_visit_time,next_ed_visit_time_diff,next_ed_visit_time_diff_days,time_to_icu_transfer_hours,triage_temperature,triage_heartrate,triage_resprate,triage_o2sat,triage_sbp,triage_dbp,triage_acuity,triage_MAP
0,98880961,98874959,82,F,White,Non-Hispanic/Non-Latino,2083-03-02 07:36:59,2083-03-02 13:44:59,2283,2083-03-02 07:36:59,2083-03-02 12:29:59,2083-03-02 12:29:59,2083-03-03 12:29:59,NaT,NaT,0.3,6.13,NaT,NaN,NaN,NaN,36.2,81.0,18.0,96.0,122.0,86.0,3,98.0
1,99121566,99354408,59,F,Other,Non-Hispanic/Non-Latino,2039-08-22 01:13:51,2039-08-22 10:45:51,2239,2039-08-22 01:16:51,2039-08-22 07:20:51,2039-08-22 07:20:51,2039-08-29 07:20:51,NaT,NaT,0.4,9.53,2040-03-10 18:13:51,4841.0,201.7,NaN,36.8,98.0,14.0,98.0,103.0,66.0,2,78.3
2,99121566,99121037,63,F,Other,Non-Hispanic/Non-Latino,2040-03-11 03:06:51,2040-03-11 13:37:51,2240,2040-03-11 03:43:51,2040-03-11 08:31:51,NaT,NaT,2040-03-11 08:31:51,2040-03-27 08:31:51,0.4,10.52,NaT,NaN,NaN,5.42,37.2,96.0,18.0,98.0,91.0,61.0,3,71.0
3,99608739,99327434,90,F,White,Non-Hispanic/Non-Latino,2096-08-31 16:43:05,2096-08-31 21:06:05,2296,2096-08-31 17:56:05,2096-08-31 20:56:05,NaT,NaT,NaT,NaT,0.2,4.38,2097-06-07 00:43:05,6704.0,279.3,NaN,NaN,69.0,16.0,98.0,107.0,42.0,3,63.7
4,99608739,99502864,89,F,White,Non-Hispanic/Non-Latino,2097-06-07 05:08:05,2097-06-07 09:35:05,2297,2097-06-07 05:59:05,2097-06-07 08:55:05,NaT,NaT,NaT,NaT,0.2,4.45,NaT,NaN,NaN,NaN,36.4,65.0,17.0,98.0,103.0,57.0,3,72.3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
118380,99648524,99789968,41,F,Other,Non-Hispanic/Non-Latino,2022-09-23 02:03:34,2022-09-23 06:38:34,2222,2022-09-23 02:58:34,2022-09-23 05:46:34,NaT,NaT,NaT,NaT,0.2,4.58,2022-09-25 21:03:34,67.0,2.8,NaN,36.8,86.0,18.0,100.0,134.0,76.0,2,95.3
118381,99312340,99391376,37,M,Black or African American,Non-Hispanic/Non-Latino,2045-12-31 02:58:40,2045-12-31 06:30:40,2245,2045-12-31 03:52:40,2045-12-31 05:10:40,2045-12-31 05:10:40,2046-01-01 05:10:40,NaT,NaT,0.1,3.53,NaT,NaN,NaN,NaN,36.9,89.0,17.0,96.0,145.0,81.0,2,102.3
118382,99828357,99836129,22,M,Other,Hispanic/Latino,2034-09-07 02:27:43,2034-09-07 06:22:43,2234,2034-09-07 04:27:43,2034-09-07 05:04:43,NaT,NaT,NaT,NaT,0.2,3.92,NaT,NaN,NaN,NaN,36.8,85.0,14.0,99.0,144.0,97.0,2,112.7
118383,98964925,98979879,24,M,Other,Hispanic/Latino,2088-11-06 10:18:50,2088-11-06 14:19:50,2288,2088-11-06 11:22:50,2088-11-06 13:11:50,NaT,NaT,NaT,NaT,0.2,4.02,NaT,NaN,NaN,NaN,NaN,75.0,16.0,100.0,122.0,78.0,4,92.7
